# ALMs — Perplexity Attribution & Token-Level Analysis (Ch.5)

This notebook is **stage 2 of the ALMs method** (Ch.5 Sec 5.3.2–5.3.3): given
the per-author GPT-2s trained by `ALMs_Train.ipynb`, decide *who wrote a
questioned document* and explain *which words drove the decision*.

**The attribution rule is argmin perplexity.** For each candidate author
we ask their ALM: "how predictable is this text to you?" The perplexity
(`PPL = exp(mean per-token NLL)`) summarises the answer; the author whose
model gives the *lowest* PPL wins.

**Three scoring stages** (each a function in `thesis_aa.alms.ppl`):

1. `score_all_pairs` — score every (ALM, test-author) pair, writing
   per-token cross-entropy logs + a PPL table;
2. `aggregate_ppl` — turn the per-token logs into one PPL per test text
   (for several text-length cutoffs);
3. `predict_and_benchmark` — attribute each test text by argmin-PPL and
   compute the benchmark metrics.

Then a fourth function, `compute_cnll`, implements the token-level
**Comparative NLL** used for interpretability (Ch.5 Eq. 3–4).

> **Provenance note.** This module is ported from the reference repo's
> `CalculatePPL.ipynb` with **7 bugs fixed** (the original did not run
> end-to-end as published) — see the `thesis_aa/alms/ppl.py` module
> docstring for the audit trail. Bug #6 (double-counted context tokens in
> overlapping sliding windows) and #7 (grouping by `text_num` alone, which
> conflated different authors' documents) are the ones that corrupted the
> original's accuracy numbers.

## 0. Bootstrap + ensure trained models exist

We need the per-author ALMs from `ALMs_Train.ipynb`. To keep this notebook
self-contained, the cell below trains them *if* `models/` is empty — with
debug hyperparameters that takes seconds. (If you already ran the training
notebook, the skip-check fires and nothing retrains.)

In [1]:
import os, sys

REPO_ROOT = os.path.dirname(os.path.abspath(os.getcwd()))  # notebooks/ -> repo root
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from thesis_aa import config, data as data_mod, eval as eval_mod
from thesis_aa.alms import train as alms_train, ppl as alms_ppl

# Always regenerate the 3-author debug corpus (the ALMs_Train demo corpus),
# regardless of what data/synthetic/ currently holds.
train_df, test_df = data_mod.generate_synthetic(
    n_authors=3, n_train_docs=6, n_test_docs=3, max_words=60, seed=0)

trained = [d for d in os.listdir(config.MODEL_DIR)
           if os.path.isdir(os.path.join(config.MODEL_DIR, d))]
if not trained:
    print('models/ is empty - training debug ALMs first (seconds)...')
    alms_train.train_all_authors(
        train_df, epochs=2, gradient_accumulation_steps=1,
        batch_size=1, block_size=64, fp16=False)
else:
    print('Found existing models:', sorted(trained))

print('device:', config.get_device(), '| train:', train_df.shape, '| test:', test_df.shape)

C:\Users\MiraMoe\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.9.0+xpu).


W0901 02:33:21.539000 28692 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Found existing models: ['author00', 'author01', 'author02']
device: xpu | train: (18, 2) | test: (9, 2)


**What you should see:** either a quick training run (three authors,
2 epochs each) or the list of already-trained models, then the synthetic
corpus shapes. Note `use_tagger=False` by default in scoring; we enable
spaCy later for the annotation demo.

## 1. Score every (model, author) pair

`score_all_pairs` builds the **n_models × n_authors scoring matrix**: cell
(i, j) answers *"how predictable is author j's test text under author i's
model?"* Each cell is a mean perplexity over that author's test documents.

Two artifacts are written per pair:

- `results/ce_log/<model_tag>-<text_tag>.csv.7z` — an LZMA-compressed CSV
  holding *per-token* data: the token strings, each token's NLL under that
  model, and (optionally) spaCy annotations. One row per test text.
- `results/ppl_result.csv` — one summary row per pair
  (`model_tag, text_tag, stride, ppl`). This file doubles as the
  **resumability log**: pairs already present are skipped on restart.

In [2]:
result_path = alms_ppl.score_all_pairs(
    train_df, test_df,
    model_dir=config.MODEL_DIR,
    use_tagger=False,          # spaCy annotations off for now (demo in step 3)
    limit_texts_per_author=None,
)
print('PPL table written to:', result_path)

[ALMs/PPL] author00 x author00


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


[ALMs/PPL] author00 x author00: PPL=687.38
[ALMs/PPL] author00 x author01
[ALMs/PPL] author00 x author01: PPL=1715.79
[ALMs/PPL] author00 x author02


[ALMs/PPL] author00 x author02: PPL=171.97


[ALMs/PPL] author01 x author00
[ALMs/PPL] author01 x author00: PPL=933.47
[ALMs/PPL] author01 x author01
[ALMs/PPL] author01 x author01: PPL=1493.18
[ALMs/PPL] author01 x author02


[ALMs/PPL] author01 x author02: PPL=180.28


[ALMs/PPL] author02 x author00
[ALMs/PPL] author02 x author00: PPL=1052.71
[ALMs/PPL] author02 x author01
[ALMs/PPL] author02 x author01: PPL=1936.60
[ALMs/PPL] author02 x author02


[ALMs/PPL] author02 x author02: PPL=167.56
PPL table written to: D:\AgentHome\Thesis\results\ppl_result.csv


**What you should see:** progress like
`[ALMs/PPL] author00 x author00: PPL=X.XX` — 3 models × 3 test authors =
9 pairs. The diagonal (own model on own texts) should tend to score lower
than off-diagonal cells — that is the entire premise of the method.

In [3]:
import pandas as pd

ppl_df = pd.read_csv(result_path)
ppl_df

,model_tag,text_tag,stride,ppl
0,author00,author00,128,687.376205
1,author00,author01,128,1715.793400
2,author00,author02,128,171.969842
3,author01,author00,128,933.469280
4,author01,author01,128,1493.176043
5,author01,author02,128,180.276862
6,author02,author00,128,1052.710314
7,author02,author01,128,1936.596448
8,author02,author02,128,167.562015


### The scoring matrix, visualised

Pivot the table so rows = ALMs, columns = true test authors. The argmin
attribution rule says: for the texts in *column j*, the predicted author is
the *row* with the lowest value.

In [4]:
matrix = ppl_df.pivot(index='model_tag', columns='text_tag', values='ppl')
print('Perplexity matrix (rows = ALM, columns = true author):')
display(matrix.round(2))
print()
print('Argmin attribution (predicted author per true author):')
print(matrix.idxmin())

Perplexity matrix (rows = ALM, columns = true author):


text_tag,author00,author01,author02
model_tag,,,
author00,687.38,1715.79,171.97
author01,933.47,1493.18,180.28
author02,1052.71,1936.60,167.56



Argmin attribution (predicted author per true author):
text_tag
author00    author00
author01    author01
author02    author02
dtype: object


**What you should see:** a 3×3 matrix whose diagonal dominates — each
author's own model finds their texts most predictable, so argmin attribution
is correct for all three authors. The synthetic lexicons are so distinct
that even 2 debug epochs separate them; on real benchmarks (100 epochs, 50
candidates) the same diagonal dominance yields the thesis's reported 88.1%
mean macro-accuracy.

## 2. Open a per-token CE log

Each `.csv.7z` archive holds the raw per-token evidence behind the matrix
cells. Reading one shows the `FEATURE_CATEGORIES` schema: the token
strings, their per-token NLLs, and (empty) annotation columns when no
tagger was used. The lists are stored *stringified* (`"['the', ...]"`)
so the CSV stays a plain CSV; `ast.literal_eval` recovers them.

In [5]:
import ast, zipfile

pair = 'author00-author00'  # author00's model, scoring author00's texts
with zipfile.ZipFile(os.path.join(config.RESULTS_DIR, 'ce_log', pair + '.csv.7z')) as z:
    inner = [n for n in z.namelist() if n.endswith('.csv')][0]
    raw = pd.read_csv(z.open(inner), names=alms_ppl.FEATURE_CATEGORIES)

print('rows (test texts):', len(raw))
row0 = raw.iloc[0]
tokens = ast.literal_eval(row0['tokens'])
losses = ast.literal_eval(row0['losses'])
per_tok = pd.DataFrame({'token': tokens[1:], 'NLL': losses})  # losses align to tokens[1:]
display(per_tok.head(12))

rows (test texts): 3


,token,NLL
0,awn,9.219748
1,were,8.259892
2,forge,14.559723
3,for,4.034334
4,he,7.741402
5,crown,8.549729
6,dawn,10.405821
7,honour,9.105698
8,throne,7.168631
9,river,9.506730


**What you should see:** a token/NLL table for the first test text.
High NLL = the model was surprised by that token. Note the list lengths:
`len(tokens) - 1 == len(losses)` — the *first* token has no preceding
context, so there is nothing to predict it; losses are aligned to
`tokens[1:]`. These per-token losses are exactly what `aggregate_ppl`
re-reads and averages into text-level PPLs — and they're the substrate for
the CNLL analysis in step 5. Common words ("the", "of") get low NLL;
author-specific lexicon words get higher NLL under other authors' models.

## 3. Linguistic annotations (spaCy, optional)

`compute_ce_per_text` can tag every GPT-2 token on the fly with spaCy
(lemma, POS, shape, …) — used in the thesis to break down attribution
evidence by linguistic category. Pass `use_tagger=True` to
`score_all_pairs`, or call the per-text function directly as below. The
spaCy model (`python -m spacy download en_core_web_sm`) is optional; when
missing, annotation columns simply stay empty and the pipeline is
unaffected.

In [6]:
tagger = None
try:
    tagger = alms_ppl._spacy_pipeline()
    print('spaCy pipeline:', tagger.__class__.__name__, '| component names:', tagger.pipe_names)
except Exception as e:
    print('spaCy unavailable - annotations will stay empty:', e)

import torch
from transformers import AutoTokenizer, GPT2LMHeadModel

device = config.get_device()
model = GPT2LMHeadModel.from_pretrained(os.path.join(config.MODEL_DIR, 'author00')).to(device)
tokenizer = AutoTokenizer.from_pretrained(os.path.join(config.MODEL_DIR, 'author00'))

sample = test_df[test_df['author_tag'] == 'author01']['text'].iloc[0]
rec_tagged = alms_ppl.compute_ce_per_text(sample, model, tokenizer, device, tagger=tagger)

if tagger is not None:
    ann = pd.DataFrame({k: rec_tagged[k] for k in ['tokens', 'lemmas', 'poss', 'tags', 'stops']})
    display(ann.head(10))
else:
    print('No tagger available - showing tokens/losses only:')
    display(pd.DataFrame({'token': rec_tagged['tokens'][:10],
                          'NLL': rec_tagged['losses'][:10]}))
del model

spaCy pipeline: English | component names: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']


,tokens,lemmas,poss,tags,stops
0,as,as,ADP,IN,True
1,binary,binary,PROPN,NNP,False
2,thread,thread,NOUN,NN,False
3,she,she,NUM,CD,False
4,kernel,kernel,NOUN,NNS,False
5,voltage,voltage,X,XX,False
6,voltage,voltage,NOUN,NN,False
7,of,of,NOUN,NNS,False
8,from,from,PROPN,NNP,False
9,they,they,PROPN,NNP,False


**What you should see:** the same token stream with filled `lemmas`,
`poss`, `tags`, `stops` columns. The thesis uses these annotations to
aggregate per-token losses by POS class or stopword status and analyse
*what kind* of tokens carry authorship signal (Ch.5 Sec 5.4).

## 4. Aggregate per-text PPL and benchmark

`aggregate_ppl` reads all CE logs back, computes one PPL per test text per
candidate, and writes a tidy dataframe. The `test_text_limits` parameter is
the thesis's **text-length sweep** (Ch.5 Sec 5.4.1): setting a limit *k*
truncates every text to its first *k* tokens before averaging, studying how
attribution accuracy grows with questioned-document length. We use a small
sweep here — `[10, 20, None]` (None = full length); the full constant lives
in `config.DEFAULT_TEST_TEXT_LIMITS`.

Then `predict_and_benchmark` attributes each `(true_tag, text_num)` group
by argmin-PPL (grouping by *both* keys — bug #7 in the original notebook
grouped by `text_num` alone and silently conflated different authors'
documents) and writes precision/recall/F1/accuracy per author.

In [7]:
ppl_paths = alms_ppl.aggregate_ppl(
    test_text_limits=[10, 20, None],   # None = full length (thesis sweep: config.DEFAULT_TEST_TEXT_LIMITS)
)
agg = pd.read_csv(ppl_paths[-1])       # the full-length buffer
print('full-length buffer:', ppl_paths[-1])
display(agg.head(8))

[ALMs/PPL] wrote D:\AgentHome\Thesis\results\ppl_dfs_buffer\ppl_dfs_buffer-10.csv
[ALMs/PPL] wrote D:\AgentHome\Thesis\results\ppl_dfs_buffer\ppl_dfs_buffer-20.csv


[ALMs/PPL] wrote D:\AgentHome\Thesis\results\ppl_dfs_buffer\ppl_dfs_buffer-full.csv
full-length buffer: D:\AgentHome\Thesis\results\ppl_dfs_buffer\ppl_dfs_buffer-full.csv


,true_tag,candidate_tag,text_num,global-ppl:(losses_shifted),global-ppl:(losses)
0,author00,author00,0,891.207521,891.207521
1,author00,author00,1,644.186927,644.186927
2,author00,author00,2,313.681085,313.681085
3,author01,author00,0,1199.110077,1199.110077
4,author01,author00,1,1569.724329,1569.724329
5,author01,author00,2,1368.747929,1368.747929
6,author02,author00,0,85.713721,85.713721
7,author02,author00,1,199.868127,199.868127


**What you should see:** the buffer with columns
`true_tag, candidate_tag, text_num, global-ppl:(losses_shifted),
global-ppl:(losses)` — for each test text, one row per candidate ALM. Two
"features" are stored: `losses` (the PPL as scored) and `losses_shifted`
(the PPL when each token's loss is attributed to the *next* position — a
one-position shift the original notebook carried for its alignment
convention; both are reported in the benchmark).

In [8]:
bench_paths = alms_ppl.predict_and_benchmark(ppl_paths)
print('benchmark CSVs:', [os.path.basename(p) for p in bench_paths])

bench = pd.read_csv(bench_paths[-1])   # full-length benchmark
print()
print('GLOBAL rows (overall metrics per feature):')
display(bench[bench['true_tag'] == 'GLOBAL'])

[ALMs/PPL] benchmark -> D:\AgentHome\Thesis\results\benchmark_results_df_home\benchmark_results_df_buffer-ppl_dfs_buffer-10.csv
[ALMs/PPL] benchmark -> D:\AgentHome\Thesis\results\benchmark_results_df_home\benchmark_results_df_buffer-ppl_dfs_buffer-20.csv
[ALMs/PPL] benchmark -> D:\AgentHome\Thesis\results\benchmark_results_df_home\benchmark_results_df_buffer-ppl_dfs_buffer-full.csv
benchmark CSVs: ['benchmark_results_df_buffer-ppl_dfs_buffer-10.csv', 'benchmark_results_df_buffer-ppl_dfs_buffer-20.csv', 'benchmark_results_df_buffer-ppl_dfs_buffer-full.csv']

GLOBAL rows (overall metrics per feature):


,feature,true_tag,fscore,precision,recall,accuracy
0,global-ppl:(losses),GLOBAL,0.885714,0.916667,0.888889,0.888889
4,global-ppl:(losses_shifted),GLOBAL,0.885714,0.916667,0.888889,0.888889


**What you should see:** for each feature (`global-ppl:(losses_shifted)`
and `global-ppl:(losses)`), a GLOBAL row plus one row per author with
fscore / precision / recall / accuracy. Even with just 2 debug epochs the
trivially-separable synthetic corpus yields ~0.89 macro-accuracy
(1.0 / 1.0 / 0.67 per author) — the pipeline works end-to-end; the
thesis's 88.1% is the same machinery at 50 candidates and 100 epochs.

In [9]:
# Thesis-style summary: macro-average over the per-author rows, per length.
summary = eval_mod.summarize_benchmark_dir(
    os.path.join(config.RESULTS_DIR, 'benchmark_results_df_home'))
display(summary)

,file,feature,macro_accuracy,macro_fscore
0,benchmark_results_df_buffer-ppl_dfs_buffer-10.csv,global-ppl:(losses),1.000000,0.0
1,benchmark_results_df_buffer-ppl_dfs_buffer-10.csv,global-ppl:(losses_shifted),1.000000,0.0
2,benchmark_results_df_buffer-ppl_dfs_buffer-20.csv,global-ppl:(losses),0.777778,0.0
3,benchmark_results_df_buffer-ppl_dfs_buffer-20.csv,global-ppl:(losses_shifted),0.777778,0.0
4,benchmark_results_df_buffer-ppl_dfs_buffer-ful...,global-ppl:(losses),0.888889,0.0
5,benchmark_results_df_buffer-ppl_dfs_buffer-ful...,global-ppl:(losses_shifted),0.888889,0.0


The summary table averages each benchmark's per-author accuracies
into one **macro-accuracy per (length, feature)** — the same aggregation
the thesis reports per dataset. On the real benchmarks this is where the
88.1% mean macro-accuracy (Ch.5 Table 8) comes from; the length sweep
rows show accuracy improving as the questioned text gets longer.

## 5. Token-level interpretability — CNLL

The final piece of ALMs is explaining *why* a document was attributed to an
author. The **Comparative NLL** (Ch.5 Eq. 3–4) of a token for candidate *a*
is:

```
CNLL(a, i) = NLL_a(i) - mean_{b != a} NLL_b(i)
```

Negative ⇒ the token is *more predictable under a's model than under the
others* ⇒ it pushes attribution toward a. Positive ⇒ it pushes away. This
is the interpretability advantage over black-box classifiers: you can point
at the exact words that decided the case.

To build the `(n_tokens-1, n_authors)` NLL matrix we run every ALM over the
same questioned text — one forward pass per model — using
`compute_ce_per_text`:

In [10]:
import numpy as np

model_tags = sorted(d for d in os.listdir(config.MODEL_DIR)
                    if os.path.isdir(os.path.join(config.MODEL_DIR, d)))
questioned = test_df.iloc[0]['text']
print('Questioned text (true author:', test_df.iloc[0]['author_tag'], ')')
print(questioned[:120], '...\n')

nll_cols, toks = {}, None
for tag in model_tags:
    m = GPT2LMHeadModel.from_pretrained(os.path.join(config.MODEL_DIR, tag)).to(device)
    tok = AutoTokenizer.from_pretrained(os.path.join(config.MODEL_DIR, tag))
    rec = alms_ppl.compute_ce_per_text(questioned, m, tok, device)
    toks = rec['tokens']
    nll_cols[tag] = rec['losses']   # unpadded: length len(tokens)-1, aligned to tokens[1:]
    del m

nll_matrix = np.column_stack([nll_cols[t] for t in model_tags])
print('NLL matrix shape (n_tokens-1, n_authors):', nll_matrix.shape)

Questioned text (true author: author00 )
<BOS>dawn were forge for he crown dawn honour throne river she kingdom for castle dawn kingdom that castle in one and in ...



NLL matrix shape (n_tokens-1, n_authors): (33, 3)


Now `compute_cnll` compares the candidate columns. With
`candidate=None` it automatically picks the **predicted author** (argmin of
total NLL) — matching how the thesis reports the evidence for whoever was
attributed. The alignment is one-to-one: row `i` of `nll_matrix` is the NLL
of `tokens[i+1]`, and `cnll[i]` belongs to the same token — so the evidence
table pairs `tokens[1:]` with `cnll`:

In [11]:
cnll = alms_ppl.compute_cnll(nll_matrix, model_tags)  # candidate=None -> predicted author
predicted = model_tags[int(np.argmin(nll_matrix.sum(axis=0)))]
print(f'Predicted author (lowest total NLL): {predicted}')
print(f'True author: {test_df.iloc[0]["author_tag"]}\n')

evidence = pd.DataFrame({'token': toks[1:], 'CNLL': cnll})
evidence = evidence.sort_values('CNLL').reset_index(drop=True)
print('Tokens most favouring', predicted, '(most negative CNLL):')
display(evidence.head(8))
print()
print('Tokens most against', predicted, '(most positive CNLL):')
display(evidence.tail(5))

Predicted author (lowest total NLL): author00
True author: author00

Tokens most favouring author00 (most negative CNLL):


,token,CNLL
0,kingdom,-2.024005
1,honour,-1.515225
2,crown,-1.329131
3,oath,-1.302324
4,banner,-1.277466
5,throne,-1.216906
6,oath,-1.214357
7,forge,-0.756555



Tokens most against author00 (most positive CNLL):


,token,CNLL
28,were,0.103395
29,in,0.163328
30,but,0.263265
31,one,0.336389
32,was,0.447314


**What you should see:** a ranked token table. Negative-CNLL tokens
are the document's authorial "fingerprints" — words the winning model
expected. On the synthetic corpus these tend to be the author's lexicon
words (e.g. *castle*, *banner* for author00); on real corpora the thesis
(Ch.5 Sec 5.4.3) shows function words and phrase-level habits carrying the
signal. Either way, attribution becomes *auditable*: you can show a human
exactly which words mattered.

## 6. Going to real data

The same three calls, on a real benchmark with 100-epoch models, reproduce
the thesis's ALMs numbers:

```python
train_df, test_df = data_mod.load_benchmark('Blogs50')   # 50 candidate authors
result_path = alms_ppl.score_all_pairs(train_df, test_df, model_dir=config.MODEL_DIR)
ppl_paths = alms_ppl.aggregate_ppl(test_text_limits=config.DEFAULT_TEST_TEXT_LIMITS)
bench_paths = alms_ppl.predict_and_benchmark(ppl_paths)
summary = eval_mod.summarize_benchmark_dir(os.path.join(config.RESULTS_DIR, 'benchmark_results_df_home'))
```

| Aspect | Demo | Thesis |
|---|---|---|
| Candidates | 3 synthetic authors | 50 real authors |
| Epochs per ALM | 2 | 100 |
| Length sweep | {10, 20, full} | 17 lengths (10–450, full) |
| Headline metric | ~chance (debug data) | 88.1% mean macro-accuracy |

For a full reproduction: train with `config.ALMS_TRAIN_CONFIG`
(`ALMs_Train.ipynb`, step 5), score with `stride=128` (default), and
aggregate with `config.DEFAULT_TEST_TEXT_LIMITS`. `score_all_pairs` is
resumable via `results/ppl_result.csv`, so a long scoring run can be
interrupted and restarted safely.